## importacao das bibliotecas do ambiente
    basicamente o "requests" permite puxar dados de um servidor
    o "path" é para facilitar a manipulacao de arquivos, tanto no windows quanto em linux
    

In [5]:
import requests
from pathlib import Path

salvamos o link da base de dados para caso a url mudar ficar mais fácil pegar novamente

In [6]:
URL_DADOS = (
    "https://s3.sa-east-1.amazonaws.com/"
    "ckan.saude.gov.br/SRAG/2019/"
    "INFLUD19-23-03-2026.csv"
)

antes de baixar os dados e arriscar baixar um arquivo assombrosamente grande verificaremos se o servidor está respondendo

In [7]:
try:
    resposta = requests.get(
        URL_DADOS,
        stream=True,
        timeout=30
    )

    print("Status HTTP:", resposta.status_code)
    print("Tamanho informado (bytes):",
          resposta.headers.get("Content-Length", "Não informado"))

    resposta.raise_for_status()
    print("Conexão estabelecida com sucesso!")

    resposta.close()

except requests.RequestException as erro:
    print("Erro ao acessar a fonte:", erro)

Status HTTP: 200
Tamanho informado (bytes): 51658150
Conexão estabelecida com sucesso!


In [8]:
import requests
import shutil
from pathlib import Path

# Localizar a raiz do repositório
raiz = next(
    pasta for pasta in [Path.cwd(), *Path.cwd().parents]
    if (pasta / "requirements.txt").exists()
    and (pasta / "notebooks").is_dir()
)

destino = raiz / "data" / "bronze"
destino.mkdir(parents=True, exist_ok=True)

# Consultar espaço disponível
espaco = shutil.disk_usage(destino)
gib = 1024 ** 3

print(f"Destino: {destino}")
print(f"Espaço livre: {espaco.free / gib:.2f} GiB")

# Consultar tamanho do CSV sem baixar seu conteúdo
resposta = requests.head(
    URL_DADOS,
    allow_redirects=True,
    timeout=30
)

if resposta.ok:
    tamanho = resposta.headers.get("Content-Length")

    if tamanho is not None:
        print(f"Tamanho do arquivo: {int(tamanho) / gib:.3f} GiB")
    else:
        print("Servidor não informou o tamanho.")
else:
    print(f"Consulta de tamanho indisponível: HTTP {resposta.status_code}")

Destino: /home/roso/Projetos/R0Z01-Projeto-3-Projeto-integrador---Group-IV/data/bronze
Espaço livre: 341.08 GiB
Tamanho do arquivo: 0.048 GiB


In [ ]:
import os

# Limites de segurança
MIB = 1024 ** 2
GIB = 1024 ** 3

LIMITE_ARQUIVO = 300 * MIB    # Máximo de 300 MiB por CSV
LIMITE_BRONZE = 2 * GIB       # Máximo de 2 GiB em CSVs Bronze
RESERVA_DISCO = 5 * GIB      # Preservar 5 GiB livres

# Começa desativado para evitar downloads acidentais
BAIXAR = True

arquivo = destino / Path(URL_DADOS).name
temporario = arquivo.with_suffix(".csv.part")

if not BAIXAR:
    print("Download desativado. Nenhum arquivo foi baixado.")

elif arquivo.exists():
    print(f"Arquivo já existe: {arquivo}")
    print("Download ignorado para evitar duplicação.")

else:
    # Espaço já utilizado pelos CSVs Bronze
    uso_bronze = sum(
        p.stat().st_size for p in destino.glob("*.csv")
    )

    if uso_bronze >= LIMITE_BRONZE:
        raise RuntimeError("Limite total da Bronze atingido.")

    try:
        with requests.get(
            URL_DADOS,
            stream=True,
            timeout=(10, 45),
            headers={"Accept-Encoding": "identity"}
        ) as resposta:

            resposta.raise_for_status()

            tamanho_header = resposta.headers.get("Content-Length")
            tamanho_previsto = (
                int(tamanho_header) if tamanho_header else None
            )

            if tamanho_previsto and tamanho_previsto > LIMITE_ARQUIVO:
                raise RuntimeError("Arquivo excede o limite permitido.")

            if tamanho_previsto and uso_bronze + tamanho_previsto > LIMITE_BRONZE:
                raise RuntimeError("Arquivo excederia o limite da Bronze.")

            if temporario.exists():
                temporario.unlink()

            baixados = 0
            ultimo_aviso = 0

            with open(temporario, "wb") as f:
                for bloco in resposta.iter_content(chunk_size=MIB):
                    if not bloco:
                        continue

                    novo_total = baixados + len(bloco)

                    if novo_total > LIMITE_ARQUIVO:
                        raise RuntimeError("Limite do arquivo atingido.")

                    if uso_bronze + novo_total > LIMITE_BRONZE:
                        raise RuntimeError("Limite total da Bronze atingido.")

                    livre = shutil.disk_usage(destino).free

                    if livre - len(bloco) < RESERVA_DISCO:
                        raise RuntimeError("Reserva de disco atingida.")

                    f.write(bloco)
                    baixados = novo_total

                    if baixados - ultimo_aviso >= 50 * MIB:
                        print(f"Baixados: {baixados / MIB:.1f} MiB")
                        ultimo_aviso = baixados

            if tamanho_previsto is not None and baixados != tamanho_previsto:
                raise RuntimeError("Download incompleto: tamanho divergente.")

            # Só confirma o arquivo após concluir a transferência
            os.replace(temporario, arquivo)

            print(f"Download concluído: {arquivo.name}")
            print(f"Tamanho salvo: {baixados / MIB:.2f} MiB")

    finally:
        # Remove arquivo parcial em caso de interrupção ou erro
        temporario.unlink(missing_ok=True)

Download desativado. Nenhum arquivo foi baixado.
